In [6]:
# A* search
import heapq
import copy

class PuzzleState:
    def __init__(self, board, g=0, h=0, parent=None):
        self.board = board
        self.g = g
        self.h = h
        self.f = g + h
        self.parent = parent

    def __lt__(self, other):
        return self.f < other.f

    def get_blank_pos(self):
        for r in range(3):
            for c in range(3):
                if self.board[r][c] == 0:
                    return r, c

    def get_neighbors(self, goal_pos_map):
        neighbors = []
        r, c = self.get_blank_pos()
        moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # Up, Down, Left, Right

        for dr, dc in moves:
            nr, nc = r + dr, c + dc
            if 0 <= nr < 3 and 0 <= nc < 3:
                new_board = copy.deepcopy(self.board)
                # Swap blank (0) with neighbor
                new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]

                h = calculate_manhattan(new_board, goal_pos_map)
                neighbors.append(PuzzleState(new_board, self.g + 1, h, self))
        return neighbors

def calculate_manhattan(board, goal_pos_map):
    distance = 0
    for r in range(3):
        for c in range(3):
            val = board[r][c]
            if val != 0:
                target_r, target_c = goal_pos_map[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance

def get_goal_map(goal):
    goal_pos_map = {}
    for r in range(3):
        for c in range(3):
            val = goal[r][c]
            if val != 0:
                goal_pos_map[val] = (r, c)
    return goal_pos_map

def solve_a_star(initial, goal):
    goal_pos_map = get_goal_map(goal)
    initial_h = calculate_manhattan(initial, goal_pos_map)
    start_state = PuzzleState(initial, 0, initial_h)

    open_set = []
    heapq.heappush(open_set, start_state)
    visited = set()

    while open_set:
        current = heapq.heappop(open_set)


        board_tuple = tuple(tuple(row) for row in current.board)
        if board_tuple in visited:
            continue
        visited.add(board_tuple)

        if current.board == goal:
            path = []
            curr = current
            while curr:
                path.append(curr.board)
                curr = curr.parent
            return path[::-1]

        for neighbor in current.get_neighbors(goal_pos_map):
            neighbor_tuple = tuple(tuple(row) for row in neighbor.board)
            if neighbor_tuple not in visited:
                heapq.heappush(open_set, neighbor)

    return None

def print_board(board):
    for row in board:
        print(" ".join(map(str, row)))
    print()


initial_state = [
    [2, 8, 3],
    [1, 6, 4],
    [7, 0, 5]
]

goal_state = [
    [1, 2, 3],
    [8, 0, 4],
    [7, 6, 5]
]


solution_path = solve_a_star(initial_state, goal_state)

if solution_path:
    print(f"Solution found in {len(solution_path) - 1} moves:\n")
    for step, board in enumerate(solution_path):
        print(f"Step {step}:")
        print_board(board)
else:
    print("No solution found.")

Solution found in 5 moves:

Step 0:
2 8 3
1 6 4
7 0 5

Step 1:
2 8 3
1 0 4
7 6 5

Step 2:
2 0 3
1 8 4
7 6 5

Step 3:
0 2 3
1 8 4
7 6 5

Step 4:
1 2 3
0 8 4
7 6 5

Step 5:
1 2 3
8 0 4
7 6 5



In [5]:
# Hill climbing
def get_conflicts(state):
    """Calculates the number of conflicting queen pairs in a given state."""
    n = len(state)
    conflicts = 0
    for i in range(n):
        for j in range(i + 1, n):
            if state[i] == state[j] or abs(state[i] - state[j]) == abs(i - j):
                conflicts += 1
    return conflicts

def find_direct_path_to_solution(initial_state):
    """Finds a direct path from the initial state to a solution state using Depth-First Search."""
    queue = [[initial_state]]
    visited = {tuple(initial_state)}

    while queue:
        path = queue.pop(0)
        current_state = path[-1]

        if get_conflicts(current_state) == 0:
            return path

        n = len(current_state)
        for col in range(n):
            for row in range(n):
                if current_state[col] == row:
                    continue
                neighbor = list(current_state)
                neighbor[col] = row
                neighbor_tuple = tuple(neighbor)

                if neighbor_tuple not in visited:
                    visited.add(neighbor_tuple)
                    queue.append(path + [neighbor])
    return None

def print_board(state):
    """Visualizes the board state."""
    n = len(state)
    for row in range(n):
        line = []
        for col in range(n):
            if state[col] == row:
                line.append("Q")
            else:
                line.append(".")
        print(" ".join(line))
    print()

# User specified initial state
initial_state_4q = [3, 1, 2, 0]

# Find the direct solution path starting from the initial state
solution_path = find_direct_path_to_solution(initial_state_4q)

if solution_path:
    print(f"Direct solution path found in {len(solution_path) - 1} steps:\n")
    for step, state in enumerate(solution_path):
        print(f"Step {step} (State: {state}, Conflicts: {get_conflicts(state)}):")
        print_board(state)
else:
    print("No path to solution found.")

Found 2 unique solution paths!

==================== SOLUTION PATH 1 (Length: 4 steps) ====================
Step 0 (State: [3, 1, 2, 0], Conflicts: 2):
. . . Q
. Q . .
. . Q .
Q . . .

Step 1 (State: [1, 1, 2, 0], Conflicts: 2):
. . . Q
Q Q . .
. . Q .
. . . .

Step 2 (State: [1, 3, 2, 0], Conflicts: 1):
. . . Q
Q . . .
. . Q .
. Q . .

Step 3 (State: [1, 3, 0, 0], Conflicts: 1):
. . Q Q
Q . . .
. . . .
. Q . .

Step 4 (State: [1, 3, 0, 2], Conflicts: 0):
. . Q .
Q . . .
. . . Q
. Q . .


==================== SOLUTION PATH 2 (Length: 4 steps) ====================
Step 0 (State: [3, 1, 2, 0], Conflicts: 2):
. . . Q
. Q . .
. . Q .
Q . . .

Step 1 (State: [2, 1, 2, 0], Conflicts: 3):
. . . Q
. Q . .
Q . Q .
. . . .

Step 2 (State: [2, 0, 2, 0], Conflicts: 2):
. Q . Q
. . . .
Q . Q .
. . . .

Step 3 (State: [2, 0, 3, 0], Conflicts: 1):
. Q . Q
. . . .
Q . . .
. . Q .

Step 4 (State: [2, 0, 3, 1], Conflicts: 0):
. Q . .
. . . Q
Q . . .
. . Q .


